# ch05 序列到序列预测（练习版）

> 补全所有 `____`，让每个代码块末尾的 `assert` 全绿。
> 数据：`data/load_curve.csv` 的 **A 台区**（8760 h），`window=72` / `horizon=24`。
>
> **每题都跟了 `# 提示：`**，照着提示能独立做完。
> 如果 assert 的数字对不上，**先回头看上一步的打印输出**，别急着改 assert —— 那些数字都是实跑出来的。
>
> ⚠️ 训练三个 Seq2Seq 大约要 1 分钟，耐心等。

## 任务清单

| 任务 | 主题 | 挖空 | 对应讲解小节 |
|---|---|---|---|
| 1 | 清洗 + 只用训练段拟合缩放参数 | 2 | §5.1 / §5.1.1 |
| 2 | 滑窗 + 三块切分（含 gap） | 2 | §5.1 |
| 3 | 单步模型 | 2 | §5.1.1 |
| 4 | 递归多步滚动 | 2 | §5.2.1 |
| 5 | 直接多输出 | 2 | §5.3 |
| 6 | Seq2Seq（全 teacher forcing） | 2 | §5.4 |
| 7 | teacher forcing 三档对照 | 3 | §5.4.1 |
| 8 | 两条朴素基线 | 2 | §5.1.2 |
| 9 | MAE–步长曲线 + 排行榜 | 1 | §5.5 |
| 10 | 误差累积量化 + 样本对照图 | 2 | §5.2.1 / §5.5.1 |

**合计 20 个挖空。**

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

DATA = Path("data")          # notebook 的 cwd = coding/05_timeseries
CSV = DATA / "load_curve.csv"

# 固定线程数：CPU 上矩阵乘法的规约顺序与线程数有关，锁死它结果才可复现
torch.set_num_threads(4)

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
np.set_printoptions(precision=4, suppress=True)

print("torch", torch.__version__, "| CUDA 可用？", torch.cuda.is_available(),
      "（本章数据小，CPU 足够）")

In [ ]:
# ---- 超参：全部写死在 SETUP 里，后面每个 cell 都只引用它们 ----
WINDOW, HORIZON = 72, 24     # 输入看过去 3 天，预测未来 1 天
SPLIT_RATIO = 0.8            # 按时间切分：前 80% 训练
HIDDEN, HIDDEN_S2S = 64, 32  # MLP 隐层 / Seq2Seq 隐层
EPOCHS, EPOCHS_S2S = 60, 20  # MLP 训练轮数 / Seq2Seq 训练轮数
LR, BATCH = 1e-2, 256        # 学习率 / batch 大小
SEED = 0
TF_LIST = (1.0, 0.5, 0.0)    # 三档 teacher_forcing_ratio

raw = pd.read_csv(CSV, encoding="utf-8-sig", parse_dates=["时间戳"])
df = raw.drop_duplicates().sort_values("时间戳").reset_index(drop=True)

print("原始行数 %d → 去重排序后 %d" % (len(raw), len(df)))
print("台区", sorted(df["台区编号"].unique()))
print("时间跨度 %s ~ %s" % (df["时间戳"].min(), df["时间戳"].max()))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# macOS 上 matplotlib 默认字体不含中文，不设置的话图上全是方块
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib", matplotlib.__version__, "| 中文字体 PingFang SC")

In [ ]:
# 脚手架：下面的常量与工具函数两版都有，你只填 @@todo 块里的内容


def clean_series(s):
    """按**小时分组**的中位数做 3 倍阈值判异常，再线性插值（与 ch01 同一口径）。

    为什么要按小时分组：工业台区是「夜间低 + 日间高」的双峰分布，
    全局中位数做阈值会把整段日间峰误判成异常。
    """
    med = s.groupby(s.index.hour).transform("median")
    return s.where(s.between(0, 3 * med)).interpolate()


class MLP1(nn.Module):
    """单步模型：吃 window 长度的窗口，吐 1 个数（下一小时）。递归多步就是反复调用它。"""

    def __init__(self, window, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(window, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x):
        return self.net(x)


class MLP24(nn.Module):
    """直接多输出：最后一层直接是 nn.Linear(hidden, horizon)，一次吐出未来 24 小时。"""

    def __init__(self, window, hidden=64, horizon=24):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(window, hidden), nn.ReLU(),
                                 nn.Linear(hidden, horizon))

    def forward(self, x):
        return self.net(x)


class Seq2Seq(nn.Module):
    """encoder-decoder：encoder 读完整个输入窗口，把 (h, c) 交给 decoder。

    decoder 每一步吃「上一时刻的值」，输出「下一时刻的值」：
      - 训练时 tf（teacher_forcing_ratio）决定这一步吃**真值**还是吃**自己的预测**
        tf=1.0 全用真值 / tf=0.0 全用预测 / 0<tf<1 按比例随机混合
      - 推理时永远不传 y，于是自动退化成「把预测喂回去」的递归解码
    这就是「训练与推理不一致」（exposure bias）的来源，见 5.4.1。
    """

    def __init__(self, hidden=32, horizon=24):
        super().__init__()
        self.horizon = horizon
        self.enc = nn.LSTM(1, hidden, batch_first=True)
        self.dec = nn.LSTM(1, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x, y=None, tf=1.0):
        if x.dim() == 2:
            x = x.unsqueeze(-1)                     # (B, T) → (B, T, 1)
        _, (h, c) = self.enc(x)
        inp = x[:, -1, :]                           # 解码的第 0 步吃「窗口最后一个真值」
        outs = []
        for t in range(self.horizon):
            o, (h, c) = self.dec(inp.unsqueeze(1), (h, c))
            p = self.head(o.squeeze(1))
            outs.append(p)
            if y is not None and tf > 0:
                keep = torch.rand(p.size(0), 1) < tf
                inp = torch.where(keep, y[:, t: t + 1], p.detach())
            else:
                inp = p.detach()
        return torch.cat(outs, dim=1)


def build(model_cls, *args, seed=SEED):
    """在固定种子下**实例化**模型。

    「权重初始化」本身就是一层随机数 —— 只在 `train_torch` 里固定种子是不够的：
    模型的初始权重是在 `train_torch` **之前**、用**全局随机源**抽出来的。
    不一起固定，跑第二遍数字就变了，`assert` 全部作废。

    顺带一个好处：三档 teacher forcing 的 Seq2Seq 会拿到**完全相同的初始权重**，
    差异只可能来自 `tf` 本身。
    """
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        return model_cls(*args)


def train_torch(model, X, Y, tf=None, epochs=60, lr=1e-2, batch=256, seed=0):
    """通用训练循环（CPU、小数据）。

    tf=None → 直接回归（模型输出 (B, horizon)）；tf=float → Seq2Seq 模式。
    另一处随机性（shuffle、teacher forcing 采样）也包在 fork_rng + manual_seed 里。
    """
    xt = torch.tensor(np.asarray(X), dtype=torch.float32)
    yt = torch.tensor(np.asarray(Y), dtype=torch.float32)
    if yt.dim() == 1:
        yt = yt[:, None]
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
        lossf = nn.MSELoss()
        n = len(xt)
        for _ in range(epochs):
            perm = torch.randperm(n)
            for k in range(0, n, batch):
                idx = perm[k: k + batch]
                opt.zero_grad()
                out = model(xt[idx]) if tf is None else model(xt[idx], yt[idx], tf)
                loss = lossf(out, yt[idx])
                loss.backward()
                if tf is not None:
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step()
    return model


def mae_per_step(pred, true):
    """逐步 MAE：返回长度 = 预测步数的数组，第 k 个数是「第 k+1 步」的 MAE。"""
    return np.abs(np.asarray(pred, dtype=np.float64)
                  - np.asarray(true, dtype=np.float64)).mean(axis=0)

## 任务 1：清洗 A 台区 + 只用训练段拟合缩放参数（§5.1）

In [ ]:
a_frame = df[df["台区编号"] == "STATION_A_01"].set_index("时间戳").sort_index()
s_a = a_frame["负荷值"].asfreq("h")

s_clean = clean_series(s_a)
S = s_clean.values.astype(np.float64)

SPLIT = int(SPLIT_RATIO * len(S))

MU = float(S[:SPLIT].mean())
SD = float(S[:SPLIT].std())

Z = (S - MU) / SD

print("整点 %d | 清洗前缺失 %d | 清洗后缺失 %d" % (len(S), int(s_a.isna().sum()), int(np.isnan(S).sum())))
print("训练段 MU %.10f / SD %.10f | 切分点 %d" % (MU, SD, SPLIT))
print("原始序列 min %.2f max %.2f（含哨兵 9999 与负值）" % (s_a.min(), s_a.max()))

# ---- 验收 ----
assert len(S) == 8760 and int(np.isnan(S).sum()) == 0
assert int(s_a.isna().sum()) == 70
assert abs(MU - 454.561801) < 1e-5, f"训练段均值实际是 {MU}"
assert abs(SD - 72.623128) < 1e-5, f"训练段标准差实际是 {SD}"

## 任务 2：滑窗与三块切分（训练 / gap / 测试）（§5.1）

In [ ]:
N = len(S) - WINDOW - HORIZON + 1
STARTS = np.arange(N)

tr_idx = STARTS[STARTS + WINDOW + HORIZON <= SPLIT]
te_idx = STARTS[STARTS >= SPLIT]

x_all = np.stack([Z[i: i + WINDOW] for i in STARTS]).astype(np.float32)
y_all = np.stack([Z[i + WINDOW: i + WINDOW + HORIZON] for i in STARTS]).astype(np.float32)

Xtr, Ytr = x_all[tr_idx], y_all[tr_idx]
Xte, Yte = x_all[te_idx], y_all[te_idx]
Xte_t = torch.tensor(Xte)
Yte_raw = np.stack([S[i + WINDOW: i + WINDOW + HORIZON] for i in te_idx])

print("样本总数 %d | 训练 %d | 测试 %d | 夹在切分点上的 gap 窗口 %d 个"
      % (N, len(tr_idx), len(te_idx), N - len(tr_idx) - len(te_idx)))
print("Xtr %s | Ytr %s | Xte %s | Yte_raw %s"
      % (Xtr.shape, Ytr.shape, Xte_t.shape, Yte_raw.shape))
print("测试段第一个预测起点 = 第 %d 个整点（切分点 %d）" % (int(te_idx[0]) + WINDOW, SPLIT))

# ---- 验收 ----
assert N == 8665
assert len(tr_idx) == 6913 and len(te_idx) == 1657
assert N - len(tr_idx) - len(te_idx) == 95, "跨在切分点上的窗口有 95 个"
assert Xtr.shape == (6913, 72) and Ytr.shape == (6913, 24) and Xte_t.shape == (1657, 72)
assert int(te_idx[0]) == SPLIT

## 任务 3：单步模型 —— 先看清「口径」的基准（§5.1.1）

In [ ]:
model_1 = build(MLP1, WINDOW, HIDDEN)

model_1 = train_torch(model_1, Xtr, Ytr[:, :1], epochs=EPOCHS, lr=LR, seed=SEED)

model_1.eval()
with torch.no_grad():
    step1 = model_1(Xte_t)[:, 0].numpy().astype(np.float64) * SD + MU
mae_1 = float(np.abs(step1 - Yte_raw[:, 0]).mean())
print("单步（horizon=1）测试 MAE = %.6f kW" % mae_1)

# ---- 验收 ----
assert abs(mae_1 - 14.544809) < 1e-5, f"单步 MAE 实际是 {mae_1}"
assert float(np.abs(step1 - Yte_raw[:, 0]).max()) < 120

## 任务 4：递归多步 —— 把预测喂回去滚动 24 次（§5.2.1）

In [ ]:
model_1.eval()
rec = np.zeros((len(te_idx), HORIZON), dtype=np.float32)
win = Xte.copy()                       # 每个测试样本的初始输入窗口

with torch.no_grad():
    for t in range(HORIZON):
        p = model_1(torch.tensor(win))[:, 0].numpy()
        win = np.concatenate([win[:, 1:], p[:, None]], axis=1)
        rec[:, t] = p

rec_raw = rec.astype(np.float64) * SD + MU

mae_rec = mae_per_step(rec_raw, Yte_raw)

print("递归多步 24 步平均 MAE = %.6f kW" % mae_rec.mean())
print("  第 1 步 %.6f → 第 12 步 %.6f → 第 24 步 %.6f"
      % (mae_rec[0], mae_rec[11], mae_rec[23]))
print("  逐步 MAE:", np.round(mae_rec, 4).tolist())

# ---- 验收 ----
assert abs(mae_rec[0] - mae_1) < 1e-9, "递归的第 1 步就是单步模型本身，必须逐位相等"
assert abs(mae_rec.mean() - 16.630607) < 1e-4, f"递归 24 步 MAE 实际是 {mae_rec.mean()}"
assert mae_rec[23] > mae_rec[0] > 0

## 任务 5：直接多输出 —— 一次吐出 24 维（§5.3）

In [ ]:
model_24 = build(MLP24, WINDOW, HIDDEN, HORIZON)

model_24 = train_torch(model_24, Xtr, Ytr, epochs=EPOCHS, lr=LR, seed=SEED)

model_24.eval()
with torch.no_grad():
    dir_raw = model_24(Xte_t).numpy().astype(np.float64) * SD + MU
mae_dir = mae_per_step(dir_raw, Yte_raw)
print("直接多输出 24 步平均 MAE = %.6f kW" % mae_dir.mean())
print("  第 1 步 %.6f → 第 24 步 %.6f" % (mae_dir[0], mae_dir[23]))
print("  逐步 MAE:", np.round(mae_dir, 4).tolist())

# ---- 验收 ----
assert abs(mae_dir.mean() - 16.696673) < 1e-4, f"直接多输出 MAE 实际是 {mae_dir.mean()}"
assert mae_dir[23] > mae_dir[0]

## 任务 6：Seq2Seq（encoder-decoder）+ 全 teacher forcing（§5.4）

In [ ]:
model_tf1 = train_torch(build(Seq2Seq, HIDDEN_S2S, HORIZON), Xtr, Ytr,
                        tf=1.0, epochs=EPOCHS_S2S, lr=LR, seed=SEED)

model_tf1.eval()
with torch.no_grad():
    tf1_raw = model_tf1(Xte_t).numpy().astype(np.float64) * SD + MU
mae_tf1 = mae_per_step(tf1_raw, Yte_raw)
print("Seq2Seq（全 teacher forcing，tf=1.0）24 步平均 MAE = %.6f kW" % mae_tf1.mean())
print("  第 1 步 %.6f → 第 24 步 %.6f" % (mae_tf1[0], mae_tf1[23]))

# ---- 验收 ----
assert abs(mae_tf1.mean() - 21.120460) < 1e-4, f"tf=1.0 的 MAE 实际是 {mae_tf1.mean()}"
assert tf1_raw.shape == (1657, 24)

## 任务 7：teacher forcing 三档对照 —— exposure bias 实测（§5.4.1）

In [ ]:
tf_curves = {1.0: mae_tf1}          # tf=1.0 的那条曲线上一格已经算好了，直接复用
for tf in (0.5, 0.0):
    m = train_torch(build(Seq2Seq, HIDDEN_S2S, HORIZON), Xtr, Ytr,
                    tf=tf, epochs=EPOCHS_S2S, lr=LR, seed=SEED)
    m.eval()
    pr = m(Xte_t).detach().numpy().astype(np.float64) * SD + MU
    tf_curves[tf] = mae_per_step(pr, Yte_raw)

print("teacher_forcing_ratio 对照（推理时一律不喂真值、只把预测喂回去）：")
for tf in TF_LIST:
    print("  tf=%.1f → 24 步平均 MAE %.6f kW（第 1 步 %.6f，第 24 步 %.6f）"
          % (tf, tf_curves[tf].mean(), tf_curves[tf][0], tf_curves[tf][23]))

# ---- 验收 ----
assert abs(tf_curves[0.5].mean() - 20.583487) < 1e-4
assert abs(tf_curves[0.0].mean() - 18.237022) < 1e-4
assert tf_curves[1.0].mean() > tf_curves[0.5].mean() > tf_curves[0.0].mean(), "tf 越大推理越差"

## 任务 8：两条朴素基线（§5.1.2）

In [ ]:
last_te = S[te_idx + WINDOW - 1]
pred_persist = np.repeat(last_te[:, None], HORIZON, axis=1)
prev24 = np.stack([S[i + WINDOW - HORIZON: i + WINDOW] for i in te_idx])

mae_persist = mae_per_step(pred_persist, Yte_raw)
mae_season = mae_per_step(prev24, Yte_raw)
print("persistence（老值延续）24 步平均 MAE = %.6f kW" % mae_persist.mean())
print("seasonal naive（昨日同时刻）24 步平均 MAE = %.6f kW" % mae_season.mean())
print("seasonal 的逐步 MAE 几乎是平的:", np.round(mae_season, 4).tolist())

# ---- 验收 ----
assert abs(mae_persist.mean() - 60.671651) < 1e-4, f"persistence 实际是 {mae_persist.mean()}"
assert abs(mae_season.mean() - 21.741549) < 1e-4, f"seasonal 实际是 {mae_season.mean()}"
assert mae_season.max() - mae_season.min() < 0.2, "seasonal 的误差不随步长累积"

## 任务 9：MAE–步长曲线 + 24 步平均 MAE 排行榜（§5.5）

In [ ]:
steps = np.arange(1, HORIZON + 1)

fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.plot(steps, mae_persist, ":", color="tab:gray", label="persistence（老值延续）")
ax.plot(steps, mae_season, "--", color="tab:green", label="seasonal naive（昨日同时刻）")
ax.plot(steps, mae_rec, "-o", ms=3.5, label="递归多步 recursive")
ax.plot(steps, mae_dir, "-s", ms=3.5, label="直接多输出 direct")
ax.plot(steps, tf_curves[0.5], "-^", ms=3.5, label="Seq2Seq（混合 TF=0.5）")
ax.axhline(mae_1, color="crimson", lw=0.9, ls="-.", label="单步 MAE（horizon=1）")
ax.set_xlabel("预测步长（小时）")
ax.set_ylabel("MAE（kW）")
ax.set_title("多步预测：MAE 随步长的变化（A 台区，window=72 / horizon=24）")
ax.set_xticks(np.arange(0, 25, 4))
ax.legend(loc="lower right", fontsize=9)

summary = {
    "persist": float(mae_persist.mean()),
    "seasonal": float(mae_season.mean()),
    "recursive": float(mae_rec.mean()),
    "direct": float(mae_dir.mean()),
    "tf1.0": float(tf_curves[1.0].mean()),
    "tf0.5": float(tf_curves[0.5].mean()),
    "tf0.0": float(tf_curves[0.0].mean()),
}

ranking = sorted(summary.items(), key=lambda kv: kv[1])

print("24 步平均 MAE 排行榜（越小越好）：")
for k, v in ranking:
    print("  %-10s %.6f kW" % (k, v))
plt.tight_layout()

# ---- 验收 ----
assert summary["recursive"] < summary["direct"] < summary["seasonal"]
assert summary["tf1.0"] > summary["tf0.5"] > summary["tf0.0"], "exposure bias 的单调性"
assert summary["seasonal"] < summary["persist"] / 2, "重复单个老值的误差是当日周期的两倍多"

## 任务 10：误差累积量化 + 单样本对照图（§5.2.1 / §5.5.1）

In [ ]:
amp = float(mae_rec.mean() / mae_1)
gap24 = float(mae_dir[23] - mae_rec[23])

err_rec = np.abs(rec_raw - Yte_raw).mean(axis=1)
err_dir = np.abs(dir_raw - Yte_raw).mean(axis=1)
k = int(np.argmax(np.abs(err_dir - err_rec)))

hor = np.arange(1, HORIZON + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.9))
axes[0].plot(hor, Yte_raw[k], "-o", ms=3, color="black", label="真实")
axes[0].plot(hor, rec_raw[k], "--s", ms=3, label="递归")
axes[0].plot(hor, dir_raw[k], "--^", ms=3, label="直接多输出")
axes[0].set_xlabel("预测步长（小时）")
axes[0].set_ylabel("负荷 kW")
axes[0].set_title("第 %d 个测试样本的 24 小时预测" % k)
axes[0].legend(fontsize=9)
axes[1].plot(hor, np.abs(rec_raw[k] - Yte_raw[k]), "--s", ms=3, label="递归")
axes[1].plot(hor, np.abs(dir_raw[k] - Yte_raw[k]), "--^", ms=3, label="直接多输出")
axes[1].set_xlabel("预测步长（小时）")
axes[1].set_ylabel("绝对误差 kW")
axes[1].set_title("该样本的逐点绝对误差")
axes[1].legend(fontsize=9)
plt.tight_layout()

print("递归 24 步平均 MAE / 单步 MAE = %.4f（误差放大 %.1f%%）"
      % (amp, (amp - 1) * 100))
print("第 24 步：递归 %.6f | 直接 %.6f | 差 %+.6f" % (mae_rec[23], mae_dir[23], gap24))
print("递归逐步增幅：第 1 步 %.4f → 第 24 步 %.4f（%+.1f%%）"
      % (mae_rec[0], mae_rec[23], (mae_rec[23] / mae_rec[0] - 1) * 100))

# ---- 验收 ----
assert abs(amp - 1.1434) < 1e-4, f"放大倍数实际是 {amp}"
assert abs(gap24 - 0.271966) < 1e-5, f"第 24 步的差实际是 {gap24}"
assert amp > 1.0, "递归一定比单步差 —— 差别就是误差累积的代价"

## 自查清单（能不看笔记答出来才算过）

- [ ] 10 个代码块的 assert 全部通过
- [ ] 单步 MAE 是多少？递归 24 步平均是多少？两者差几个百分点？
- [ ] 切分处的 `gap` 有多少个样本？为什么必须丢掉？
- [ ] 递归滚动时窗口为什么要 `[:, 1:]`？不丢会怎样？
- [ ] 为什么 `mae_rec[0]` 必须严格等于单步 MAE？
- [ ] 「误差累积」的机制一句话怎么说？递归第 24 步的 MAE 相对第 1 步涨了多少？
- [ ] 直接多输出为什么「没有累积」，却还是输给了递归？
- [ ] `teacher_forcing_ratio` 三档里，哪一档推理 MAE 最差？这叫什么现象？三档的排序是单调的吗？
- [ ] `tf=0.0` 训练时最大的风险是什么？本章靠什么压住的？
- [ ] 两条朴素基线各是多少？哪一条是本章的"及格线"？
- [ ] 「U 形」的曲线对应哪条方法？它为什么两头低？
- [ ] 本章为什么用 MAE 而不是 MSE 做评估指标？

## 关键真值对照（做完再对）

| 量 | 值 |
|---|---|
| 切分点 `SPLIT` | 7008 |
| 训练 / gap / 测试窗口数 | 6913 / **95** / 1657 |
| 训练段 `MU` / `SD` | 454.561801 / 72.623128 |
| 单步 `horizon=1` MAE | **14.544809** |
| 递归 24 步平均 MAE | **16.630607** |
| 递归第 1 / 12 / 24 步 MAE | 14.5448 / 16.7367 / 18.0992 |
| 直接多输出 24 步平均 MAE | **16.696673** |
| 直接第 1 / 24 步 MAE | 14.8118 / 18.3712 |
| Seq2Seq `tf=1.0` / `0.5` / `0.0` | 21.120460 / 20.583487 / **18.237022** |
| persistence 基线 | 60.671651（U 形：30.1→81.3→21.8） |
| seasonal naive 基线 | **21.741549**（一条平线） |
| 递归放大倍数 `mae_rec.mean()/mae_1` | **1.1434** |
| 第 24 步「直接 − 递归」 | **+0.271966 kW** |

> **选模型的口诀**：先摆基线（21.74）→ 再看曲线形状 → 最后才比平均 MAE。
> **永远用推理模式的指标选模型**，不看训练 loss。